In [7]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)
    
MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
API_KEY=os.getenv('API_KEY')

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=API_KEY)

In [30]:
system_prompt = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [31]:
system_prompt += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [34]:
def chat(message,history):
    history=[{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_prompt=system_prompt
    if 'belt' in message.lower():
        relevant_system_prompt+="The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    messages=[{"role":"system", "content":relevant_system_prompt}]+history+[{"role":"user", "content":message}]
    stream=ollama.chat.completions.create(model=MODEL,messages=messages,stream=True)
    response=""
    for chunk in stream:
        response+=chunk.choices[0].delta.content or ' '
        yield response

In [35]:
gr.ChatInterface(fn=chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
